# Deploy a Joblib Model to a Managed Online Endpoint

1. Replace `model/customer_model.joblib` with the trusted customer model.
2. Replace `input.csv` with rows matching that model's feature order.
3. Edit the Azure values in Cell 2.
4. Run Cell 3 to load the model and score the CSV locally.
5. Set `DEPLOY = True` and run Cell 4 to deploy and score the same rows in Azure.

Joblib can execute code during deserialization. Only load a model received through a trusted and verified channel, and update `environment/conda.yaml` to match its training environment.

**Source:** Adapted from Microsoft's [Azure ML `model-1` online endpoint example](https://github.com/Azure/azureml-examples/tree/main/sdk/python/endpoints/online/model-1), licensed under the [MIT License](https://github.com/Azure/azureml-examples/blob/main/LICENSE). See the original [score.py](https://github.com/Azure/azureml-examples/blob/main/sdk/python/endpoints/online/model-1/onlinescoring/score.py), [sample model](https://github.com/Azure/azureml-examples/blob/main/sdk/python/endpoints/online/model-1/model/sklearn_regression_model.pkl), and [sample request](https://github.com/Azure/azureml-examples/blob/main/sdk/python/endpoints/online/model-1/sample-request.json).

In [ ]:
SUBSCRIPTION_ID = "<subscription-id>"
TENANT_ID = "<tenant-id>"
RESOURCE_GROUP = "<resource-group>"
WORKSPACE_NAME = "<workspace-name>"

MODEL_NAME = "customer-joblib-model"
ENVIRONMENT_NAME = "customer-joblib-environment"
ENDPOINT_NAME = "customer-joblib-endpoint"
DEPLOYMENT_NAME = "blue"
INSTANCE_TYPE = "Standard_DS3_v2"

DEPLOY = False

In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd

ROOT = Path(__vsc_ipynb_file__).resolve().parent
MODEL_PATH = ROOT / "model/customer_model.joblib"
INPUT_PATH = ROOT / "input.csv"
REQUEST_PATH = ROOT / "sample-request.json"

model = joblib.load(MODEL_PATH)
input_data = pd.read_csv(INPUT_PATH)
local_predictions = model.predict(input_data.to_numpy())

results = input_data.copy()
results["prediction"] = local_predictions
display(results)

request = {"data": input_data.values.tolist()}
REQUEST_PATH.write_text(json.dumps(request, indent=2), encoding="utf-8")
print(f"Model: {MODEL_PATH.name}")
print(f"Scored rows: {len(results)}")

/anaconda/envs/azureml-workshop/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 0.24.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,prediction
0,1,2,3,4,5,6,7,8,9,10,11055.977246
1,10,9,8,7,6,5,4,3,2,1,4503.079536


Model: customer_model.joblib
Scored rows: 2


In [ ]:
if DEPLOY:
    from azure.ai.ml import MLClient
    from azure.ai.ml.constants import AssetTypes
    from azure.ai.ml.entities import (
        CodeConfiguration,
        Environment,
        ManagedOnlineDeployment,
        ManagedOnlineEndpoint,
        Model,
    )
    from azure.identity import AzureCliCredential

    ml_client = MLClient(
        AzureCliCredential(tenant_id=TENANT_ID),
        SUBSCRIPTION_ID,
        RESOURCE_GROUP,
        WORKSPACE_NAME,
    )

    registered_model = ml_client.models.create_or_update(
        Model(
            name=MODEL_NAME,
            path=str(MODEL_PATH),
            type=AssetTypes.CUSTOM_MODEL,
        )
    )
    registered_environment = ml_client.environments.create_or_update(
        Environment(
            name=ENVIRONMENT_NAME,
            image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
            conda_file=str(ROOT / "environment/conda.yaml"),
        )
    )

    endpoint = ml_client.online_endpoints.begin_create_or_update(
        ManagedOnlineEndpoint(name=ENDPOINT_NAME, auth_mode="key")
    ).result()

    deployment = ml_client.online_deployments.begin_create_or_update(
        ManagedOnlineDeployment(
            name=DEPLOYMENT_NAME,
            endpoint_name=ENDPOINT_NAME,
            model=registered_model,
            environment=registered_environment,
            code_configuration=CodeConfiguration(
                code=str(ROOT / "onlinescoring"),
                scoring_script="score.py",
            ),
            instance_type=INSTANCE_TYPE,
            instance_count=1,
        )
    ).result()

    response = ml_client.online_endpoints.invoke(
        endpoint_name=ENDPOINT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        request_file=str(REQUEST_PATH),
    )
    cloud_predictions = json.loads(response)
    np.testing.assert_allclose(cloud_predictions, local_predictions)
    print(f"Cloud predictions: {cloud_predictions}")

    endpoint.traffic = {DEPLOYMENT_NAME: 100}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    print(f"Traffic routed to {DEPLOYMENT_NAME}")
else:
    print("Set DEPLOY = True in Cell 2 when you are ready to deploy.")

## Expected Result

The trusted joblib model predicts locally, Azure ML registers immutable model and environment versions, the managed deployment reaches `Succeeded`, and cloud invocation returns predictions for `sample-request.json`.